## tl;dr

- 已检查 10 张 CSV；质量规则 **24/24 通过**。
- 知识库初始语料含 **7 份文档、116 页、116 个可追溯切片**。
- 推荐演示场景：银黄口服液 2026-05、板蓝根颗粒 2026-05、六味地黄胶囊 2026-03。
- 原料表缺少独立采购价和实物消耗量，价格/数量效应不可被精确拆分；系统必须把市场行情仅作为佐证而非确定性因果。

## Context & Methods

目标是验证模拟数据能否支撑环比、同比、预算差异、工厂对标、三要素贡献度、原料/费用下钻和知识库检索。

### Key Assumptions

- 金额单位以文件列名为准，月份按自然月，北京时间。
- 单位成本贡献率只在总单位成本变化绝对值不小于 0.005 元/盒时解释。
- 赛题模拟数据是比赛计算的控制来源；外部资料只做法规、方法和行业知识增强。

## Data

输入来自赛题 V1.1 净化解压副本；清洗版统一写为 UTF-8-SIG，便于 Windows Excel 正确显示中文。

In [1]:
from pathlib import Path
import pandas as pd

project = Path(r'''D:\项目\2026年第二节重庆市AI大模型创新应用大赛\企业赛道_创灵境_基于RAG与大模型的制药企业产品成本智能分析报告系统''')
profile = pd.read_csv(project / '06_评测/数据文件质量概览.csv', encoding='utf-8-sig')
quality = pd.read_csv(project / '06_评测/数据一致性检查.csv', encoding='utf-8-sig')
monthly = pd.read_csv(project / '01_数据/02_分析输出/产品月度成本指标.csv', encoding='utf-8-sig')
profile[['file', 'rows_after', 'columns_after', 'exact_duplicate_rows', 'null_cells']]

,file,rows_after,columns_after,exact_duplicate_rows,null_cells
0,中药一厂_人工工时明细_2026年1-6月.csv,18,9,0,0
1,中药一厂_制造费用明细_2026年1-6月.csv,90,8,0,0
2,中药一厂_原材料消耗明细_2026年1-6月.csv,108,9,0,0
3,中药一厂_成本汇总_2025年1-6月.csv,18,10,0,0
4,中药一厂_成本汇总_2026年1-6月.csv,18,10,0,0
5,中药一厂_预算数据_2026年.csv,18,10,0,0
6,中药二厂_成本汇总_2025年1-6月.csv,18,10,0,0
7,中药二厂_成本汇总_2026年1-6月.csv,18,10,0,0
8,药材市场价格行情_2026年上半年.csv,13,11,0,0
9,行业成本基准数据_2026.csv,15,7,0,0


## Results

### 1. 数据一致性

In [2]:
quality

,check,status,severity,metric,tolerance,implication
0,中药一厂_成本汇总_2025年1-6月.csv: 三要素合计=单位成本,PASS,none,0.0,≤ 0.011,成本结构、瀑布图和贡献度的数值底座
1,中药一厂_成本汇总_2025年1-6月.csv: 单位成本×产量=总成本,PASS,none,0.0,≤ 1.0,总成本汇总与报告金额
2,中药一厂_成本汇总_2026年1-6月.csv: 三要素合计=单位成本,PASS,none,0.0,≤ 0.011,成本结构、瀑布图和贡献度的数值底座
3,中药一厂_成本汇总_2026年1-6月.csv: 单位成本×产量=总成本,PASS,none,0.0,≤ 1.0,总成本汇总与报告金额
4,中药二厂_成本汇总_2025年1-6月.csv: 三要素合计=单位成本,PASS,none,0.0,≤ 0.011,成本结构、瀑布图和贡献度的数值底座
5,中药二厂_成本汇总_2025年1-6月.csv: 单位成本×产量=总成本,PASS,none,0.0,≤ 1.0,总成本汇总与报告金额
6,中药二厂_成本汇总_2026年1-6月.csv: 三要素合计=单位成本,PASS,none,0.0,≤ 0.011,成本结构、瀑布图和贡献度的数值底座
7,中药二厂_成本汇总_2026年1-6月.csv: 单位成本×产量=总成本,PASS,none,0.0,≤ 1.0,总成本汇总与报告金额
8,预算三要素合计=预算单位成本,PASS,none,0.0,≤ 0.011,预算差异分析
9,预算单位成本×预算产量=预算总成本,PASS,none,0.0,≤ 1.0,预算总额分析


### 2. 三个高信息量演示场景

In [3]:
scenarios = pd.read_csv(project / '01_数据/02_分析输出/推荐三场景指标.csv', encoding='utf-8-sig')
scenarios[['产品名称','月份','单位成本(元/盒)','环比变动率','同比变动率','预算偏差率','对标差异率']]

,产品名称,月份,单位成本(元/盒),环比变动率,同比变动率,预算偏差率,对标差异率
0,六味地黄胶囊,2026-03,17.02,-0.032955,0.005910,-0.004678,-0.063806
1,板蓝根颗粒,2026-05,7.47,0.031768,0.052113,0.067143,-0.062735
2,银黄口服液,2026-05,11.21,0.028440,0.046685,0.057547,-0.033621


### 3. 知识语料构建状态

In [4]:
manifest = pd.read_csv(project / '02_知识库/03_索引/document_manifest.csv', encoding='utf-8-sig')
manifest[['source_file','pages','characters','chunks','ocr_required','document_type']]

,source_file,pages,characters,chunks,ocr_required,document_type
0,GMP法规核心摘要_2010修订版.pdf,7,3554,7,False,法规制度
1,产品配方文档_六味地黄胶囊.pdf,3,1346,3,False,产品配方
2,产品配方文档_板蓝根颗粒.pdf,2,866,2,False,产品配方
3,产品配方文档_银黄口服液.pdf,2,961,2,False,产品配方
4,生产工艺文档_中药一厂.pdf,5,2578,5,False,生产工艺
5,药品生产质量管理规范GMP.pdf,93,34014,93,False,法规制度
6,车间设备清单_中药一厂.pdf,4,2179,4,False,设备台账


## Takeaways

1. 汇总表与材料、人工、制造费用明细可形成可复验的确定性计算链。
2. 应以三场景黄金集覆盖价格行情、工艺/设备事件和工厂对标。
3. 大模型不得承担加减乘除、连接或阈值判断；它只接收已验证 JSON 和带页码证据。
4. 后续应为每个场景人工标注关键事实、允许引用和禁止断言，形成检索与生成双层评测集。